# Ddpg Actor Critic

PyTorch actor/critic networks, replay memory and a DDPG training loop.

Stage: prototype.


In [ ]:
import time
import numpy as np
import torch
import torch.nn.functional as F
import gym
from collections import deque
import random

class ActorNetwork(torch.nn.Module):
    def __init__(self, state_dim, action_dim, max_action):
        super(ActorNetwork, self).__init__()
        self.l1 = torch.nn.Linear(state_dim, 400)
        self.l2 = torch.nn.Linear(400, 300)
        self.l3 = torch.nn.Linear(300, action_dim)
        self.max_action = max_action

    def forward(self, state):
        a = F.relu(self.l1(state))
        a = F.relu(self.l2(a))
        return self.max_action * torch.tanh(self.l3(a))

class CriticNetwork(torch.nn.Module):
    def __init__(self, state_dim, action_dim):
        super(CriticNetwork, self).__init__()
        self.l1 = torch.nn.Linear(state_dim + action_dim, 400)
        self.l2 = torch.nn.Linear(400, 300)
        self.l3 = torch.nn.Linear(300, 1)

    def forward(self, state, action):
        q = F.relu(self.l1(torch.cat([state, action], 1)))
        q = F.relu(self.l2(q))
        return self.l3(q)

class ReplayBuffer:
    def __init__(self, max_size):
        self.buffer = deque(maxlen=max_size)

    def add(self, transition):
        self.buffer.append(transition)

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        return state, action, reward, next_state, done

    def size(self):
        return len(self.buffer)

class DDPGAgent:
    def __init__(self, state_dim, action_dim, max_action):
        self.actor = ActorNetwork(state_dim, action_dim, max_action).to(device)
        self.actor_target = ActorNetwork(state_dim, action_dim, max_action).to(device)
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=1e-4)

        self.critic = CriticNetwork(state_dim, action_dim).to(device)
        self.critic_target = CriticNetwork(state_dim, action_dim).to(device)
        self.critic_target.load_state_dict(self.critic.state_dict())
        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=1e-3)

        self.replay_buffer = ReplayBuffer(max_size=1000000)
        self.max_action = max_action
        self.discount = 0.99
        self.tau = 0.005

        self.critic_loss_history = []
        self.actor_loss_history = []
        self.q_value_history = []

    def select_action(self, state):
        state = torch.FloatTensor(state.reshape(1, -1)).to(device)
        return self.actor(state).cpu().data.numpy().flatten()

    def train(self, batch_size=64):
        if self.replay_buffer.size() < batch_size:
            return

        state, action, reward, next_state, done = self.replay_buffer.sample(batch_size)
        state = torch.FloatTensor(state).to(device)
        action = torch.FloatTensor(action).to(device)
        reward = torch.FloatTensor(reward).to(device)
        next_state = torch.FloatTensor(next_state).to(device)
        done = torch.FloatTensor(done).to(device)

        # Critic update
        target_Q = self.critic_target(next_state, self.actor_target(next_state))
        target_Q = reward + (1 - done) * self.discount * target_Q
        current_Q = self.critic(state, action)
        critic_loss = F.mse_loss(current_Q, target_Q)

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        # Actor update
        actor_loss = -self.critic(state, self.actor(state)).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # Soft update
        for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

        for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

        # Store loss and Q value for plotting
        self.critic_loss_history.append(critic_loss.item())
        self.actor_loss_history.append(actor_loss.item())
        self.q_value_history.append(current_Q.mean().item())

    def add_transition(self, transition):
        self.replay_buffer.add(transition)

def train_ddpg(env, agent, episodes, max_timesteps, batch_size):
    episode_rewards = []
    print_intervals = [2 ** i for i in range(int(np.log2(episodes)) + 1)]  # Exponential intervals for printing

    start_time = time.time()

    for episode in range(episodes):
        state = env.reset()
        episode_reward = 0

        for t in range(max_timesteps):
            action = agent.select_action(state)
            next_state, reward, done, _ = env.step(action)
            agent.add_transition((state, action, reward, next_state, done))
            state = next_state
            episode_reward += reward

            agent.train(batch_size)

            if done:
                break

        episode_rewards.append(episode_reward)

        if episode + 1 in print_intervals:
            elapsed_time = time.time() - start_time
            print(f"Episode: {episode + 1}, Reward: {episode_reward}, "
                  f"Critic Loss: {np.mean(agent.critic_loss_history)}, "
                  f"Actor Loss: {np.mean(agent.actor_loss_history)}, "
                  f"Average Q Value: {np.mean(agent.q_value_history)}, "
                  f"Time Elapsed: {elapsed_time:.2f} seconds")

    return episode_rewards

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    env = gym.make('Pendulum-v1')  # Replace with your environment

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    max_action = float(env.action_space.high[0])

    agent = DDPGAgent(state_dim, action_dim, max_action)
    episodes = 1000  # Number of episodes
    max_timesteps = 200  # Maximum timesteps per episode
    batch_size = 64  # Batch size for training

    train_ddpg(env, agent, episodes, max_timesteps, batch_size)
